### Load Data & BM25 Model

In [1]:
import pandas as pd
import numpy as np
import pickle
import math

print("Loading dataset and BM25 model...")
df = pd.read_csv('../../data_resource/preprocessed_recipes.csv')
df['clean_text'] = df['clean_text'].fillna('')

with open('../models/bm25_model.pkl', 'rb') as f:
    bm25 = pickle.load(f)

print(f"Loaded {len(df)} recipes and BM25 model successfully.")

Loading dataset and BM25 model...
Loaded 522517 recipes and BM25 model successfully.


### Define Evaluation Metrics

In [2]:
print("Initializing evaluation functions (Precision, Recall, NDCG)...")

def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    relevant_retrieved = set(retrieved_k).intersection(set(relevant))
    return len(relevant_retrieved) / k

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    retrieved_k = retrieved[:k]
    relevant_retrieved = set(retrieved_k).intersection(set(relevant))
    return len(relevant_retrieved) / len(relevant)

def dcg_at_k(retrieved, relevance_scores, k):
    dcg = 0.0
    for i, idx in enumerate(retrieved[:k]):
        rel = relevance_scores.get(idx, 0)
        dcg += (2**rel - 1) / math.log2(i + 2)
    return dcg

def ndcg_at_k(retrieved, relevance_scores, k):
    actual_dcg = dcg_at_k(retrieved, relevance_scores, k)

    ideal_indices = sorted(relevance_scores.keys(), key=lambda x: relevance_scores[x], reverse=True)
    ideal_dcg = dcg_at_k(ideal_indices, relevance_scores, k)

    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

print("Evaluation functions are ready.")

Initializing evaluation functions (Precision, Recall, NDCG)...
Evaluation functions are ready.


### Mock Query & Calculate Scores

In [3]:
query = "spicy chicken noodle"
tokenized_query = query.lower().split()
k = 5

print(f"Starting evaluation simulation for query: '{query}'")

scores = bm25.get_scores(tokenized_query)
top_indices = np.argsort(scores)[::-1][:10]

print("\n--- Top Retrieved Results ---")
for i, idx in enumerate(top_indices[:k], 1):
    print(f"Rank {i}: {df.iloc[idx]['Name']}")

mock_relevance_scores = {
    top_indices[0]: 3,
    top_indices[1]: 3,
    top_indices[2]: 2,
    top_indices[3]: 0,
    top_indices[4]: 1,
}

relevant_indices = [idx for idx, score in mock_relevance_scores.items() if score > 0]

p_at_k = precision_at_k(top_indices, relevant_indices, k)
r_at_k = recall_at_k(top_indices, relevant_indices, k)
ndcg_val = ndcg_at_k(top_indices, mock_relevance_scores, k)

print(f"\n--- Evaluation Metrics @ K={k} ---")
print(f"Precision@{k}: {p_at_k:.4f}")
print(f"Recall@{k}:    {r_at_k:.4f}")
print(f"NDCG@{k}:      {ndcg_val:.4f}")
print("--------------------------------")
print("Evaluation completed. The metrics are ready for your report!")

Starting evaluation simulation for query: 'spicy chicken noodle'

--- Top Retrieved Results ---
Rank 1: Spicy Shoyu Ramen Noodle
Rank 2: Spicy Asian Chicken Noodle Salad
Rank 3: Spicy Noodle Salad With Chicken and Mint
Rank 4: Spicy Asian Chicken and Noodle Salad
Rank 5: Chicken Noodle Chicken

--- Evaluation Metrics @ K=5 ---
Precision@5: 0.8000
Recall@5:    1.0000
NDCG@5:      0.9967
--------------------------------
Evaluation completed. The metrics are ready for your report!
